## NoteBook to explore the conversion between the gathered data catelouge into a time series data frame for Model Training and Testing

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc
from enum import Enum
CURRENT_DIR = Path.cwd()
EARTHQUAKE_DATASET_PATH = CURRENT_DIR / "data" /"processed"/ "earthquake_data_vs30.csv"
NEPAL_DISTRICTS_DATASET_PATH = CURRENT_DIR / "data"/"important"/"district.csv"
TABLE_1_DATASET_PATH = CURRENT_DIR / "data"/"processed"/"table_1.csv"

In [2]:
class EarthquakeDataColumns(Enum):
	ID = "eq_id"
	TIME = "time"
	MAGNITUDE = "mag"
	DEPTH = "depth"
	VS30 = "vs30"
	LATITUDE = "e_latitude"
	LONGITUDE = "e_longitude"

ALL_EARTHQUAKE_COLUMNS = [col.value for col in EarthquakeDataColumns]



eq_df = pd.read_csv(EARTHQUAKE_DATASET_PATH)
eq_df["time"] = pd.to_datetime(
    eq_df["time"],
    format="mixed",
    utc=True
)
eq_df.rename(columns={"depth_km":"depth", "id":"eq_id", "latitude":"e_latitude", "longitude":"e_longitude"}, inplace=True)
eq_df = eq_df[ALL_EARTHQUAKE_COLUMNS]
eq_df.head()

,eq_id,time,mag,depth,vs30,e_latitude,e_longitude
0,iscgem799042,1970-02-12 01:51:48.800000+00:00,5.40,15.2,900.0000,29.276,81.648
1,iscgem799449,1970-02-26 19:30:06.160000+00:00,5.41,10.0,900.0000,27.653,85.914
2,iscgem793928,1970-07-21 15:37:43.570000+00:00,5.24,15.0,679.3701,27.977,84.760
3,iscgem793996,1970-07-25 01:35:24.970000+00:00,5.39,20.0,219.9635,25.595,88.704
4,iscgem784033,1971-05-03 00:33:25.850000+00:00,5.50,19.7,260.5655,30.791,84.382


In [3]:
class NepalDistrictColumns(Enum):
	ID = "d_id"
	PROVINCE = "province"
	DISTRICT = "district"
	CENTROID_LAT = "d_centroid_lat"
	CENTROID_LONG = "d_centroid_long"

ALL_NEPAL_DISTRICT_COLUMNS = [col.value for col in NepalDistrictColumns]


nd_df = pd.read_csv(NEPAL_DISTRICTS_DATASET_PATH)
nd_df.rename(columns={"id":"d_id", "centroid_lat":"d_centroid_lat", "centroid_long":"d_centroid_long"}, inplace=True)

nd_df = nd_df[ALL_NEPAL_DISTRICT_COLUMNS]

nd_df.head(5)

,d_id,province,district,d_centroid_lat,d_centroid_long
0,1,Koshi,Taplejung,27.589802,87.824840
1,2,Koshi,Panchthar,27.138090,87.787212
2,3,Koshi,Ilam,26.875623,87.901106
3,4,Koshi,Jhapa,26.593646,87.907970
4,5,Koshi,Morang,26.603388,87.475446


## External Tooling

### PGA Look Up Class

In [4]:
class PGALookUp:
	"""A class to look up PGA values based on MMI levels."""
	_pga_look_up ={
    2.0: 0.0023977,
    2.5: 0.003294,
    3.0: 0.005528164,
    3.5: 0.01043,
    4.0: 0.01856609,
    4.5: 0.027905,
    5.0: 0.031231,
    5.5: 0.060817,
    6.0: 0.065221751,
    6.5: 0.10134499899999999,
    7.0: 0.13697,
    7.5: 0.212045,
    8.0: 0.28004
}
	_lowest_mmi = min(_pga_look_up.keys())
	_highest_mmi = max(_pga_look_up.keys())
	@staticmethod
	def get_pga(mmi:float) -> float:
		"""Returns the median PGA value (in g) corresponding to a given MMI level.
		"""
		
		lookup = PGALookUp()._pga_look_up
		if mmi in lookup:
			return lookup[mmi]
		else:
			raise ValueError(f"MMI value {mmi} not found in lookup table.")
print(PGALookUp.get_pga(5.0))  # Example usage

0.031231


### Radius Model 

In [5]:
# Piecewise empirical regression model trained on USGS active-region data.
#
# Regression form:
#
#   log10(r_hypo) =
#       a
#       + b * log10(PGA)
#       + c * magnitude
#       + d * log10(VS30 / 760)
#       + e * log10(depth)
#
# Magnitude bins:
#
#   Model 1: M <= 5.0
#   Model 2: M > 5.0
#
# Test-set performance:
#
#   M <= 5.0
#       MSE  = 0.041502
#       RMSE = 0.203720
#       MAE  = 0.155973
#
#   M > 5.0
#       MSE  = 0.022438
#       RMSE = 0.149793
#       MAE  = 0.112091
#
# Final felt radius used by STEPPS:
#
#   r_epi = sqrt(r_hypo² - depth²)
#
# Planned district impact probability:
#
#   p_affect =
#       1 - Φ(
#           (log10(d_epi) - log10(r_epi))
#           / sigma
#       )
#
# Candidate sigma values:
#
#   sigma_small_mag = 0.203720
#   sigma_large_mag = 0.149793
#
# Notes:
#
#   - Trained on active tectonic-region observations.
#   - Uses PGA as the intensity measure.
#   - Produces an equivalent isotropic epicentral felt radius.
#   - Does not model rupture geometry, directivity, or path effects.

epsilon = 1e-8

def get_r_epi(
    pga: float,
    magnitude: float,
    vs_30: float,
    depth: float
) -> float:
    """
    Estimate epicentral distance (r_epi) from earthquake magnitude,
    site conditions, focal depth, and a target Peak Ground
    Acceleration (PGA).

    Parameters
    ----------
    pga : float
        Peak Ground Acceleration in units of g.

    magnitude : float
        Earthquake magnitude.

    vs_30 : float
        Average shear-wave velocity in the upper 30 meters (m/s).

    depth : float
        Earthquake focal depth (km).

    Returns
    -------
    float
        Estimated epicentral felt radius (km). Returns 0.0 if the
        predicted hypocentral distance is smaller than the focal depth.
    """

    log_pga = np.log10(pga + epsilon)
    log_vs_30 = np.log10((vs_30 + epsilon) / 760.0)
    log_depth = np.log10(depth + epsilon)

    if magnitude <= 5.0:
        a = -1.0077478524404035
        b = -0.32296656
        c = 0.43818554
        d = 0.04913303
        e = 0.07811176
    else:
        a = -0.473103270179728
        b = -0.37554408
        c = 0.28561204
        d = -0.09830627
        e = 0.0714735

    rhypo = 10 ** (
        a
        + b * log_pga
        + c * magnitude
        + d * log_vs_30
        + e * log_depth
    )

    sqr_r_epi = rhypo**2 - depth**2

    r_epi = np.sqrt(sqr_r_epi) if sqr_r_epi > 0.0 else 0.0

    return r_epi


print("Using Median PGA Lookup:")
print(
    get_r_epi(
        pga=PGALookUp.get_pga(2.5),
        magnitude=4.9,
        vs_30=900,
        depth=60.65
    )
)  # Expected: ~105.07 km

Using Median PGA Lookup:
105.06828972737411


In [6]:
def get_r_epi_vectorized(
	pga: float,
	magnitude: np.ndarray,
	vs_30: np.ndarray,
	depth: np.ndarray
) -> np.ndarray:
	"""
	Estimate epicentral distances (r_epi) for multiple locations using
	a single PGA threshold.

	Parameters
	----------
	pga : float
		Peak Ground Acceleration (g) used for all observations.
		Typically obtained from a target MMI-to-PGA lookup.

	magnitude : np.ndarray
		Earthquake magnitudes.

	vs_30 : np.ndarray
		Average shear-wave velocities in the upper 30 meters (m/s).

	depth : np.ndarray
		Earthquake focal depths (km).

	Returns
	-------
	np.ndarray
		Estimated epicentral distances (km), one per observation.
	"""
	magnitude = np.asarray(magnitude, dtype=np.float64)
	vs_30 = np.asarray(vs_30, dtype=np.float64)
	depth = np.asarray(depth, dtype=np.float64)

	log_pga = np.log10(pga + epsilon)
	log_vs_30 = np.log10((vs_30 + epsilon) / 760.0)
	log_depth = np.log10(depth + epsilon)

	mask = magnitude <= 5.0

	a = np.where(mask, -1.0077478524404035, -0.473103270179728)
	b = np.where(mask, -0.32296656, -0.37554408)
	c = np.where(mask,  0.43818554,  0.28561204)
	d = np.where(mask,  0.04913303, -0.09830627)
	e = np.where(mask,  0.07811176,  0.0714735)

	rhypo = 10 ** (
		a
		+ b * log_pga
		+ c * magnitude
		+ d * log_vs_30
		+ e * log_depth
	)
	sqr_r_epi = rhypo**2 - depth**2
	return np.sqrt(np.maximum(sqr_r_epi, 0.0))

In [7]:
def haversine_distance_vectorized(
	lat1: np.ndarray,
	lon1: np.ndarray,
	lat2: np.ndarray,
	lon2: np.ndarray
) -> np.ndarray:
	"""
	Calculate the great-circle distance between two points on the Earth
	using the Haversine formula.

	Parameters
	----------
	lat1 : np.ndarray
		Latitude of the first point(s) in decimal degrees.

	lon1 : np.ndarray
		Longitude of the first point(s) in decimal degrees.

	lat2 : np.ndarray
		Latitude of the second point(s) in decimal degrees.

	lon2 : np.ndarray
		Longitude of the second point(s) in decimal degrees.

	Returns
	-------
	np.ndarray
		Great-circle distance(s) in kilometers.
	"""
	# Convert decimal degrees to radians
	lat1_rad = np.radians(lat1)
	lon1_rad = np.radians(lon1)
	lat2_rad = np.radians(lat2)
	lon2_rad = np.radians(lon2)

	# Haversine formula
	dlat = lat2_rad - lat1_rad
	dlon = lon2_rad - lon1_rad

	a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
	c = 2 * np.arcsin(np.sqrt(a))

	# Radius of Earth in kilometers (mean radius)
	radius_earth_km = 6371.0

	return radius_earth_km * c

In [8]:
from scipy.stats import norm
import numpy as np

SIGMA_SMALL_MAG = 0.203720
SIGMA_LARGE_MAG = 0.149793

def get_p_affected(
    magnitude: np.ndarray,
    d_epi: np.ndarray,
    r_epi: np.ndarray,
) -> np.ndarray:
    """
    Compute district impact probability using the exceedance model:

        p_affect = 1 - Φ(
            (log10(d_epi) - log10(r_epi)) / sigma
        )

    where sigma is selected according to earthquake magnitude.
    """
    magnitude = np.asarray(magnitude, dtype=np.float64)
    d_epi = np.asarray(d_epi, dtype=np.float64)
    r_epi = np.asarray(r_epi, dtype=np.float64)

    sigma = np.where(
        magnitude <= 5.0,
        SIGMA_SMALL_MAG,
        SIGMA_LARGE_MAG,
    )

    z = (
        np.log10(d_epi + 1e-8)
        - np.log10(r_epi + 1e-8)
    ) / sigma
    return norm.sf(z)  # equivalent to 1 - norm.cdf(z)

## Table 1 — Earthquake–District Impact Table

This table stores probabilistic earthquake impact records for each district. It is generated once from the earthquake catalog and cached for downstream forecasting tasks.

Only district–MMI combinations with a minimum impact probability threshold (currently 0.5%) are retained to reduce noise and storage requirements.

| Column        | Description                                                                                             |
| ------------- | ------------------------------------------------------------------------------------------------------- |
| `eq_id`       | Unique identifier of the earthquake event.                                                              |
| `event_time`  | Timestamp of the earthquake occurrence.                                                                 |
| `district_id` | Unique identifier of the district.                                                                      |
| `mmi`         | Target Modified Mercalli Intensity (MMI) threshold.                                                     |
| `pga`         | Median PGA value corresponding to the selected MMI threshold.                                           |
| `mag`         | Earthquake magnitude.                                                                                   |
| `depth`       | Earthquake focal depth (km).                                                                            |
| `vs_30`       | Average shear-wave velocity in the upper 30 meters (m/s).                                               |
| `d_epi`       | Epicentral distance between the earthquake epicenter and the district centroid (km).                    |
| `r_epi`       | Predicted epicentral radius corresponding to the selected MMI threshold.                                |
| `p_affected`  | Probability that the district experienced shaking equal to or greater than the specified MMI threshold. |

### Impact Probability

For each earthquake and MMI threshold, an equivalent epicentral radius (`r_epi`) is estimated from:

* Magnitude
* Depth
* VS30
* PGA corresponding to the target MMI

District impact probability is then computed as:

$$
p_{affected}
=
1-\Phi
\left(
\frac{\log_{10}(d_{epi})-\log_{10}(r_{epi})}
{\sigma}
\right)
$$

where:

* \(d_{epi}\) is the district epicentral distance.
* \(r_{epi}\) is the predicted epicentral radius.
* \(\sigma\) is the regression residual standard deviation.
* \(\Phi\) is the standard normal cumulative distribution function.

Interpretation:

* \(p_{affected} \approx 1.0\) indicates the district was very likely affected at or above the specified MMI threshold.
* \(p_{affected} \approx 0.5\) indicates the district lies near the estimated impact boundary.
* \(p_{affected} \approx 0.0\) indicates the district was unlikely to experience that level of shaking.

---

## Table 2 — District Impact Time-Series Table

This table is generated dynamically from Table 1 and serves as the training dataset for district-level earthquake impact forecasting.

Each row represents a district on a specific date, together with historical impact activity and future prediction targets.

| Column        | Description                                                                  |
| ------------- | ---------------------------------------------------------------------------- |
| `date`        | Reference date of the observation.                                           |
| `district_id` | Unique identifier of the district.                                           |
| `cnt_7d`      | Weighted count of district-impact events during the previous 7 days.         |
| `cnt_30d`     | Weighted count of district-impact events during the previous 30 days.        |
| `cnt_90d`     | Weighted count of district-impact events during the previous 90 days.        |
| `cnt_180d`    | Weighted count of district-impact events during the previous 180 days.       |
| `y_7d`        | Weighted count of district-impact events occurring within the next 7 days.   |
| `y_30d`       | Weighted count of district-impact events occurring within the next 30 days.  |
| `y_90d`       | Weighted count of district-impact events occurring within the next 90 days.  |
| `y_180d`      | Weighted count of district-impact events occurring within the next 180 days. |

The count features are derived from Table 1 and can be computed in multiple ways depending on the forecasting objective:

* **Binary counts:** count every qualifying impact event equally.
* **Probability-weighted counts:** sum `p_affected` values.
* **Intensity-weighted counts:** weight events using both MMI and `p_affected`.

The historical columns (`cnt_*`) are used as model inputs, while the future columns (`y_*`) serve as prediction targets.


--- 
### Table 1 DataSet Creation by using the eq_df and nd_df.

In [9]:
MMIS = [2.5,4,5,6] 
PGAS = [PGALookUp.get_pga(mmi) for mmi in MMIS]
print(PGAS)  # Expected: [0.003294, 0.01856609, 0.031231, 0.065221751]

[0.003294, 0.01856609, 0.031231, 0.065221751]



### Create a Cartesian product (cross join) between:

  1. Earthquake events
  2. Nepal districts
  3. Target MMI thresholds

This produces one row for every:

  (earthquake, district, MMI)

combination.

Example:

  Earthquake A
      × Taplejung
      × MMI 2.5

  Earthquake A
      × Taplejung
      × MMI 4.0

  Earthquake A
      × Panchthar
      × MMI 2.5

  ...

The resulting table is the foundation of STEPPS Table 1 and is used
to compute:

  - r_epi      : predicted felt radius for the selected MMI threshold
  - d_epi      : epicentral distance between the earthquake and district
  - p_affected : soft district-impact probability

Expected row count:

  n_earthquakes × n_districts × n_mmi_levels

For the current dataset:

  2,045 earthquakes
  × 77 districts
  × 4 MMI levels
  = 629,860 rows

In [10]:
AFFECTED_PROBABILITY_THRESHOLD = 0.01  #1% minimum impact probability
MMIS = [2.5, 4.0, 5.0, 6.0]
PGAS = [PGALookUp.get_pga(mmi) for mmi in MMIS]



pgas_df = pd.DataFrame({
    "pga": PGAS,
    "mmi": MMIS
})

event_district_mmi_df = (
    eq_df[ALL_EARTHQUAKE_COLUMNS]
    .merge(
        nd_df[ALL_NEPAL_DISTRICT_COLUMNS],
        how="cross"
    )
    .merge(
        pgas_df,
        how="cross"
    )
)

event_district_mmi_df["r_epi"] = get_r_epi_vectorized(
	pga=event_district_mmi_df["pga"].values,
	magnitude=event_district_mmi_df["mag"].values,
	vs_30=event_district_mmi_df["vs30"].values,
	depth=event_district_mmi_df["depth"].values
)
event_district_mmi_df["d_epi"] = haversine_distance_vectorized(
	lat1=event_district_mmi_df["e_latitude"].values,
	lon1=event_district_mmi_df["e_longitude"].values,
	lat2=event_district_mmi_df["d_centroid_lat"].values,
	lon2=event_district_mmi_df["d_centroid_long"].values
)
event_district_mmi_df["p_affected"] = get_p_affected(
    magnitude=event_district_mmi_df["mag"].values,
    d_epi=event_district_mmi_df["d_epi"].values,
    r_epi=event_district_mmi_df["r_epi"].values,
)

event_district_mmi_df = event_district_mmi_df[
    event_district_mmi_df["p_affected"] >= AFFECTED_PROBABILITY_THRESHOLD
]
table_1_df = event_district_mmi_df[
    [
        "eq_id",
        "time",
        "d_id",
        "mmi",
        "pga",
        "mag",
        "depth",
        "vs30",
        "d_epi",
        "r_epi",
        "p_affected",
        
    ]
].copy()
print(event_district_mmi_df.shape)
print("Head:\n",table_1_df.head(20))
print("Sample 20:\n",table_1_df.sample(20))
table_1_df.to_csv(TABLE_1_DATASET_PATH, index=False)
print(f"Table 1 saved to {TABLE_1_DATASET_PATH}")
table_1_df

(57841, 17)
Head:
             eq_id                             time  d_id  mmi       pga  mag  \
140  iscgem799042 1970-02-12 01:51:48.800000+00:00    36  2.5  0.003294  5.4   
144  iscgem799042 1970-02-12 01:51:48.800000+00:00    37  2.5  0.003294  5.4   
148  iscgem799042 1970-02-12 01:51:48.800000+00:00    38  2.5  0.003294  5.4   
152  iscgem799042 1970-02-12 01:51:48.800000+00:00    39  2.5  0.003294  5.4   
168  iscgem799042 1970-02-12 01:51:48.800000+00:00    43  2.5  0.003294  5.4   
172  iscgem799042 1970-02-12 01:51:48.800000+00:00    44  2.5  0.003294  5.4   
176  iscgem799042 1970-02-12 01:51:48.800000+00:00    45  2.5  0.003294  5.4   
188  iscgem799042 1970-02-12 01:51:48.800000+00:00    48  2.5  0.003294  5.4   
192  iscgem799042 1970-02-12 01:51:48.800000+00:00    49  2.5  0.003294  5.4   
196  iscgem799042 1970-02-12 01:51:48.800000+00:00    50  2.5  0.003294  5.4   
200  iscgem799042 1970-02-12 01:51:48.800000+00:00    51  2.5  0.003294  5.4   
204  iscgem799042 197

,eq_id,time,d_id,mmi,pga,mag,depth,vs30,d_epi,r_epi,p_affected
140,iscgem799042,1970-02-12 01:51:48.800000+00:00,36,2.5,0.003294,5.4,15.200,900.0000,258.679033,118.887617,0.012100
144,iscgem799042,1970-02-12 01:51:48.800000+00:00,37,2.5,0.003294,5.4,15.200,900.0000,217.440154,118.887617,0.040022
148,iscgem799042,1970-02-12 01:51:48.800000+00:00,38,2.5,0.003294,5.4,15.200,900.0000,194.897315,118.887617,0.075914
152,iscgem799042,1970-02-12 01:51:48.800000+00:00,39,2.5,0.003294,5.4,15.200,900.0000,251.691453,118.887617,0.014832
168,iscgem799042,1970-02-12 01:51:48.800000+00:00,43,2.5,0.003294,5.4,15.200,900.0000,250.780571,118.887617,0.015231
...,...,...,...,...,...,...,...,...,...,...,...
629840,us6000u0hz,2026-10-06 16:53:12.578000+00:00,73,2.5,0.003294,5.1,17.838,731.2563,132.243669,100.000980,0.208897
629844,us6000u0hz,2026-10-06 16:53:12.578000+00:00,74,2.5,0.003294,5.1,17.838,731.2563,164.978488,100.000980,0.073322
629848,us6000u0hz,2026-10-06 16:53:12.578000+00:00,75,2.5,0.003294,5.1,17.838,731.2563,204.471289,100.000980,0.019053
629852,us6000u0hz,2026-10-06 16:53:12.578000+00:00,76,2.5,0.003294,5.1,17.838,731.2563,196.726055,100.000980,0.024895
